# Data Formats: SFT and RL (GSM8K)

Примеры форматов данных для SFT и RL: instruct/chat для SFTDataset, GSM8K для GRPO.

## SFT: Instruct format (JSONL)
Одна строка — один пример. Поля: `instruction`, опционально `input`, `output`.

In [ ]:
import json

instruct_examples = [
    {"instruction": "Сколько будет 2+2?", "output": "2+2=4."},
    {"instruction": "Переведи на английский: Привет.", "input": "Привет.", "output": "Hello."},
]
for ex in instruct_examples:
    print(json.dumps(ex, ensure_ascii=False))

## SFT: Chat format (messages)
Одна строка — один диалог. Поле `messages`: список `{ "role": "user"|"assistant"|"system", "content": "..." }`.

In [ ]:
chat_examples = [
    {
        "messages": [
            {"role": "user", "content": "What is the capital of France?"},
            {"role": "assistant", "content": "The capital of France is Paris."},
        ]
    },
]
print(json.dumps(chat_examples[0], indent=2, ensure_ascii=False))

## SFTDataset: как указать формат
- **instruct**: `sft_columns={'format': 'instruct', 'instruction': 'instruction', 'output': 'output'}`
- **chat**: `sft_columns={'format': 'chat', 'messages': 'messages'}` и при необходимости `chat_template` в токенизаторе.

In [ ]:
from homellm.training.sft import SFTDataset
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained('gpt2')
if tok.pad_token is None:
    tok.add_special_tokens({'pad_token': '<|pad|>'})

# Пример для instruct-файла
# ds = SFTDataset('/app/datasets/sft_train.jsonl', tok, seq_len=512, sft_columns={'format': 'instruct', 'instruction': 'instruction', 'output': 'output'})
print('SFTDataset ready for instruct or chat JSONL.')

## RL: GSM8K и load_gsm8k
Математические задачи для GRPO. `load_gsm8k(split, max_samples, reasoning_format, dataset_key)`.

In [ ]:
from homellm.training.rl.data.gsm8k import load_gsm8k, GSM8KDataset

# Загрузка с HuggingFace (требует datasets)
try:
    ds = load_gsm8k(split='train', max_samples=10, reasoning_format='deepseek')
    print('GSM8K samples:', len(ds))
    if len(ds) > 0:
        print('Example prompt:', (ds[0].prompt or '')[:200])
except Exception as e:
    print('Load error (need HF datasets):', e)

In [ ]:
# RLSample / SimplePromptDataset: свои RL датасеты
from homellm.training.rl.data.base import RLSample, SimplePromptDataset

data = [{"question": "What is 3+5?", "answer": "8"}, {"question": "2*4?", "answer": "8"}]
simple_ds = SimplePromptDataset(data, prompt_key="question", answer_key="answer")
print('Samples:', len(simple_ds), '-', simple_ds.samples[0].prompt)